## System First Light Performance construction paper system timing and dynamics: telescope slew rate

Author: Laura Toribio San Cipriano

In [1]:
import numpy as np
import scipy as sp
import matplotlib.pyplot as plt
import pandas as pd

from lsst.summit.utils.efdUtils import makeEfdClient, getEfdData, getDayObsEndTime, getDayObsStartTime
from lsst.summit.utils.tmaUtils import TMAEventMaker

from astropy.time import Time, TimeDelta
from scipy.interpolate import UnivariateSpline

import os
from datetime import datetime, timezone, timedelta

from lsst.summit.utils.tmaUtils import (
    getCommandsDuringEvent,
    TMAEvent,
    TMAEventMaker,
    TMAState,
)

# Defining parameters

In [2]:
# included for quick reference
# define limits from science requirements document (LTS-103 2.2.2) for plotting
# units in deg/s - deg/s^2 - deg/s^3
el_limit_dict = {
    "max_velocity": 5.25,
    "max_acceleration": 5.25,
    "max_jerk": 21,
    "design_velocity": 3.5,
    "design_acceleration": 3.5,
    "design_jerk": 14,
}
az_limit_dict = {
    "max_velocity": 10.5,
    "max_acceleration": 10.5,
    "max_jerk": 42,
    "design_velocity": 7,
    "design_acceleration": 7,
    "design_jerk": 28,
}

In [3]:
begin_day_obs = 20250618
end_day_obs = 20250621

In [4]:
# Create an EFD client instance
client = makeEfdClient()
eventMaker = TMAEventMaker()

# Define the start and end time
start = getDayObsStartTime(begin_day_obs)
end = getDayObsEndTime(end_day_obs)

In [5]:
# Information about az and el
az = await client.select_time_series('lsst.sal.MTMount.azimuth', \
                                            ['actualPosition', 'actualVelocity', 'timestamp', 'actualJerk', 'actualAcceleration'],  start, end)
el = await client.select_time_series('lsst.sal.MTMount.elevation', \
                                            ['actualPosition', 'actualVelocity', 'timestamp', 'actualJerk', 'actualAcceleration'],  start, end)

In [6]:
# Information about slews

# Function to generate de range of obs days
def generate_day_obs_range(start_day_obs, end_day_obs):
    start_date = datetime.strptime(str(start_day_obs), "%Y%m%d")
    end_date = datetime.strptime(str(end_day_obs), "%Y%m%d")
    current = start_date
    day_obs_list = []
    while current <= end_date:
        day_obs_list.append(int(current.strftime("%Y%m%d")))
        current += timedelta(days=1)
    return day_obs_list

In [7]:
# Slews and tracks for days
all_events = []
for day_obs in generate_day_obs_range(begin_day_obs, end_day_obs):
    print(f"Fetching events for dayObs={day_obs}")
    daily_events = eventMaker.getEvents(day_obs, addBlockInfo=True)
    all_events.extend(daily_events)

slews = [e for e in all_events if e.type == TMAState.SLEWING]
tracks = [e for e in all_events if e.type == TMAState.TRACKING]

print(f"Found {len(slews)} slews and {len(tracks)} tracks between {begin_day_obs} and {end_day_obs}")

Fetching events for dayObs=20250618


Fetching events for dayObs=20250619


Fetching events for dayObs=20250620


Fetching events for dayObs=20250621


Found 1430 slews and 1308 tracks between 20250618 and 20250621


In [ ]:
# Match between slews and az and el information
az['timestamp'] = az['timestamp'].astype(float)
el['timestamp'] = el['timestamp'].astype(float)

az_slew_rows = []
el_slew_rows = []

for idx, slew in enumerate(slews):
    start_time = slew.begin.unix
    end_time = slew.end.unix

    mask_az = (az['timestamp'] >= start_time) & (az['timestamp'] <= end_time)
    az_subset = az[mask_az].copy()

    mask_el = (el['timestamp'] >= start_time) & (el['timestamp'] <= end_time)
    el_subset = el[mask_el].copy()
    
    if not az_subset.empty:
        az_subset['slew_index'] = idx
        az_subset['slew_dayObs'] = slew.dayObs
        az_slew_rows.append(az_subset)

    if not el_subset.empty:
        el_subset['slew_index'] = idx
        el_subset['slew_dayObs'] = slew.dayObs
        el_slew_rows.append(el_subset)

az_slews = pd.concat(az_slew_rows, ignore_index=False)
el_slews = pd.concat(el_slew_rows, ignore_index=False)

print(f"az_slews contains {len(az_slews)} rows corresponding to {az_slews['slew_index'].nunique()} slews.")
print(f"el_slews contains {len(el_slews)} rows corresponding to {el_slews['slew_index'].nunique()} slews.")

# PLOTS

In [ ]:
# Distribution of slews
az_slews['start_datetime'] = pd.to_datetime(az_slews['timestamp'], unit='s')

az_slews['date'] = az_slews['start_datetime'].dt.date
az_slews['hour'] = az_slews['start_datetime'].dt.hour

# Grouped
grouped = az_slews.groupby(['date', 'hour']).size().reset_index(name='count')

all_dates = pd.date_range(az_slews['start_datetime'].dt.floor('D').min(),
                          az_slews['start_datetime'].dt.floor('D').max(),
                          freq='D').date

all_hours = np.arange(24)
full_index = pd.MultiIndex.from_product([all_dates, all_hours], names=['date', 'hour'])
grouped_full = grouped.set_index(['date', 'hour']).reindex(full_index, fill_value=0).reset_index()

grouped_full['x_label'] = grouped_full['date'].astype(str) + '\n' + grouped_full['hour'].astype(str) + 'h'

# New directory
os.makedirs("slew_plots", exist_ok=True)

fig, ax = plt.subplots(figsize=(16, 6))
bars = ax.bar(grouped_full.index, grouped_full['count'], color='steelblue', edgecolor='black')

step = 6
xticks_to_show = np.arange(0, len(grouped_full), step)
ax.set_xticks(xticks_to_show)
ax.set_xticklabels(grouped_full.loc[xticks_to_show, 'x_label'], rotation=90, fontsize=8)

ax.set_title('Date Distribution of Slew')
ax.set_xlabel('Date')
ax.set_ylabel('# Slews')
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()
fig.savefig("slew_plots/slews_hist.png", dpi=300)


In [ ]:
# Distribution of max
az_slews_ = az_slews

def extract_max_metrics(df, axis_name):
    grouped = df.groupby("slew_index")
    return pd.DataFrame({
        f"max_{axis_name}_velocity": grouped["actualVelocity"].apply(lambda x: np.max(x)),
        f"max_{axis_name}_acceleration": grouped["actualAcceleration"].apply(lambda x: np.max(x)),
        f"max_{axis_name}_jerk": grouped["actualJerk"].apply(lambda x: np.max(x)),
        f"start_time": grouped["timestamp"].min(),
        f"end_time": grouped["timestamp"].max(),
    }).reset_index()

az_metrics = extract_max_metrics(az_slews, "az")
el_metrics = extract_max_metrics(el_slews, "el")

slew_metrics = az_metrics.merge(el_metrics, on="slew_index", suffixes=('_az', '_el'))

slew_metrics["start_datetime"] = pd.to_datetime(slew_metrics["start_time_az"], unit="s")
start_date = slew_metrics["start_datetime"].min().date()
end_date = slew_metrics["start_datetime"].max().date()
num_slews = len(slew_metrics)

slew_metrics = slew_metrics[
    (slew_metrics["max_az_jerk"] <= 100) &
    (slew_metrics["max_az_acceleration"] <= 100) &
    (slew_metrics["max_el_jerk"] <= 100) &
    (slew_metrics["max_el_acceleration"] <= 100)
]

fig, axs = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle(
    f"MT Mount Vels Accels and Jerks\n{num_slews} slews from {start_date} to {end_date}",
    fontsize=16
)

def draw_limits(ax, design_val, max_val):
    ax.axvline(design_val, color='orange', linestyle='--', label='Design limit')
    ax.axvline(max_val, color='red', linestyle='--', label='Max limit')


# Azimut
axs[0, 0].hist(slew_metrics['max_az_velocity'], bins=30, color='steelblue', edgecolor='black')
draw_limits(axs[0, 0], az_limit_dict["design_velocity"], az_limit_dict["max_velocity"])
axs[0, 0].set_title('Max Velocity (Az)')
axs[0, 0].set_xlabel('Velocity (deg/s)')
axs[0, 0].set_ylabel('# slews')

axs[0, 1].hist(slew_metrics['max_az_acceleration'], bins=30, color='steelblue', edgecolor='black')
draw_limits(axs[0, 1], az_limit_dict["design_acceleration"], az_limit_dict["max_acceleration"])
axs[0, 1].set_title('Max Acceleration (Az)')
axs[0, 1].set_xlabel('Acceleration (deg/s²)')
#axs[0, 1].set_xlim(0, 20)

axs[0, 2].hist(slew_metrics['max_az_jerk'], bins=30, color='steelblue', edgecolor='black')
draw_limits(axs[0, 2], az_limit_dict["design_jerk"], az_limit_dict["max_jerk"])
axs[0, 2].set_title('Max jerk (Az)')
axs[0, 2].set_xlabel('Jerk (deg/s³)')
#axs[0, 2].set_xlim(0, 20)

# Elevación (fila inferior)
axs[1, 0].hist(slew_metrics['max_el_velocity'], bins=30, color='seagreen', edgecolor='black')
draw_limits(axs[1, 0], el_limit_dict["design_velocity"], el_limit_dict["max_velocity"])
axs[1, 0].set_title('Max Velocity (El)')
axs[1, 0].set_xlabel('Velocity (deg/s)')
axs[1, 0].set_ylabel('# slews')

axs[1, 1].hist(slew_metrics['max_el_acceleration'], bins=30, color='seagreen', edgecolor='black')
draw_limits(axs[1, 1], el_limit_dict["design_acceleration"], el_limit_dict["max_acceleration"])
axs[1, 1].set_title('Max Acceleration (El)')
axs[1, 1].set_xlabel('Acceleration (deg/s²)')

axs[1, 2].hist(slew_metrics['max_el_jerk'], bins=30, color='seagreen', edgecolor='black')
draw_limits(axs[1, 2], el_limit_dict["design_jerk"], el_limit_dict["max_jerk"])
axs[1, 2].set_title('Max jerk (El)')
axs[1, 2].set_xlabel('Jerk (deg/s³)')

for ax in axs.flat:
    ax.grid(True, linestyle='--', alpha=0.5)
    ax.legend(loc='upper right')
    
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()
fig.savefig("slew_plots/slews_max.png", dpi=300)

az_slews = az_slews_

## Slew over the limit

In [ ]:
az_slews[az_slews["actualAcceleration"]>30]

In [ ]:
def exceeds_limits(row, limit_dict, axis='az'):
    return (
        row[f'max_{axis}_velocity'] > limit_dict['max_velocity']
        or row[f'max_{axis}_acceleration'] > limit_dict['max_acceleration']
        or row[f'max_{axis}_jerk'] > limit_dict['max_jerk']
    )

In [ ]:
def plot_slew_subplot(timestamps, velocities, positions, accelerations, jerks,
                      limits_dict, axis_label, slew_idx, slew_time):
    fig, axs = plt.subplots(4, 1, figsize=(10, 8), sharex=True)
    
    spline_func = UnivariateSpline(timestamps, velocities, s=0.1)  # puedes ajustar s según el ruido

    smoothed_velocity = spline_func(timestamps)
    residual = velocities - smoothed_velocity
    
    # Velocity
    axs[0].plot(timestamps, velocities, '-', color='black', label='Actual Velocity')
    axs[0].plot(timestamps, smoothed_velocity, 'o', color='green', markersize=2, label='Smoothed Velocity')
    axs[0].axhline(limits_dict['design_velocity'], color='orange', linestyle='--', label='Design limit')
    axs[0].axhline(-limits_dict['design_velocity'], color='orange', linestyle='--')
    axs[0].axhline(limits_dict['max_velocity'], color='red', linestyle='--', label='Max limit')
    axs[0].axhline(-limits_dict['max_velocity'], color='red', linestyle='--')
    axs[0].set_ylabel('Velocity (deg/s)')
    axs[0].grid(True)

    # Position residuals (opcional)
    axs[1].plot(timestamps, residual, 'o', color='green', markersize=2, label='Residual')
    axs[1].set_ylabel('Residual (deg/s)')
    axs[1].grid(True)
    
    # Acceleration
    axs[2].plot(timestamps, accelerations, '-', color='black', label='Acceleration')
    axs[2].axhline(limits_dict['design_acceleration'], color='orange', linestyle='--')
    axs[2].axhline(-limits_dict['design_acceleration'], color='orange', linestyle='--')
    axs[2].axhline(limits_dict['max_acceleration'], color='red', linestyle='--')
    axs[2].axhline(-limits_dict['max_acceleration'], color='red', linestyle='--')
    axs[2].set_ylabel('Acceleration (deg/s²)')
    axs[2].grid(True)

    # Jerk
    axs[3].plot(timestamps, jerks, '-', color='black', label='Jerk')
    axs[3].axhline(limits_dict['design_jerk'], color='orange', linestyle='--')
    axs[3].axhline(-limits_dict['design_jerk'], color='orange', linestyle='--')
    axs[3].axhline(limits_dict['max_jerk'], color='red', linestyle='--')
    axs[3].axhline(-limits_dict['max_jerk'], color='red', linestyle='--')
    axs[3].set_ylabel('Jerk (deg/s³)')
    axs[3].set_xlabel('Timestamp (s)')
    axs[3].grid(True)

    # Save
    fecha = slew_time.strftime('%Y-%m-%dT%H-%M-%S')
    fig.suptitle(f"Slew {slew_idx} on {fecha} - {axis_label}", fontsize=14)
    plt.tight_layout(rect=[0, 0, 1, 0.96])

    filename = f"{fecha}-slew-{slew_idx:03d}-{axis_label.lower()}.png"
    output_dir = "slew_plots"
    os.makedirs(output_dir, exist_ok=True)
    plt.savefig(os.path.join(output_dir, filename))
    plt.close()


In [ ]:
az_exceed_count = 0
el_exceed_count = 0

for index, row in slew_metrics.iterrows():
    slew_index = row['slew_index']
    slew_time = row['start_datetime'] 

    # --- Azimuth ---
    if exceeds_limits(row, az_limit_dict, axis='az'):
        az_exceed_count += 1
        slew_data = az_slews[az_slews['slew_index'] == slew_index]
        

        plot_slew_subplot(
            slew_data['timestamp'].values,
            slew_data['actualVelocity'].values,
            slew_data['actualPosition'].values,
            slew_data['actualAcceleration'].values,
            slew_data['actualJerk'].values,
            az_limit_dict,
            "Azimuth",
            slew_index,
            slew_time,
        )

    # --- Elevation ---
    if exceeds_limits(row, el_limit_dict, axis='el'):
        el_exceed_count += 1
        slew_data = el_slews[el_slews['slew_index'] == slew_index]

        plot_slew_subplot(
            slew_data['timestamp'].values,
            slew_data['actualVelocity'].values,
            slew_data['actualPosition'].values,
            slew_data['actualAcceleration'].values,
            slew_data['actualJerk'].values,
            el_limit_dict,
            "Elevation",
            slew_index,
            slew_time,
        )

total_exceed = az_exceed_count + el_exceed_count
print(f"\nThere are {total_exceed} slews that exceed the telescope limits.")
print(f"  • {az_exceed_count} in Azimuth")
print(f"  • {el_exceed_count} in Elevation")

In [ ]:
************************************

In [ ]:
plt.subplots_adjust(wspace=0.5)
plt.subplot(1,2,1)
plt.hist(slew_times)
plt.xlabel("Slew and settle time (seconds)")
plt.xlim(0.0, 10.0)
plt.subplot(1,2,2)
plt.scatter(slew_dist, slew_times)
plt.ylabel("Slew and settle time(sec)")
plt.xlabel("Slew distance (degrees)")
